# Training FakeVideoDetection Pipeline

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../src"))
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from models.fusion import MultiModalFusionModel
from data.dataset import FakeVideoDataset
from tqdm.notebook import tqdm

In [ ]:
# Set random seed for consistent train/val split across notebooks
torch.manual_seed(42)

csv_path = "../dataset/metadata.csv"
batch_size = 8

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
dataset = FakeVideoDataset(csv_path, num_frames=16)
print(f"Total videos found: {len(dataset)}")

if len(dataset) > 0:
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    print(f"Train size: {train_size}, Validation size: {val_size}")

In [ ]:
# Initialize Full Model
# MultiModalFusionModel instantiates vit_backbone, temporal, frequency, audio
model = MultiModalFusionModel(vit_pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss()

class FrameLevelViT(nn.Module):
    """Temporary wrapper to train the ViT backbone independently on single frames."""
    def __init__(self, vit_extractor, dropout=0.3):
        super().__init__()
        self.vit = vit_extractor
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.vit.embed_dim, 1)
        )
    def forward(self, x):
        emb = self.vit(x)
        return self.classifier(emb)

In [ ]:
def train_stage(model_obj, dataloader, optimizer, criterion_obj, device_obj, stage_name, is_frame_level=False):
    model_obj.train()
    total_loss, correct, total = 0, 0, 0
    
    pbar = tqdm(dataloader, desc=f"Training {stage_name}")
    for batch in pbar:
        rgb = batch['rgb'].to(device_obj)
        label = batch['label'].to(device_obj).unsqueeze(1)
        
        optimizer.zero_grad()
        
        if is_frame_level:
            S = rgb.shape[1]
            frame_idx = torch.randint(0, S, (1,)).item()
            outputs = model_obj(rgb[:, frame_idx, :, :, :])
        else:
            fft, audio = batch['fft'].to(device_obj), batch['audio'].to(device_obj)
            outputs = model_obj(rgb, fft, audio)
            
        loss = criterion_obj(outputs, label)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.sigmoid(outputs) > 0.5
        correct += (preds == label).sum().item()
        total += label.size(0)
        
        pbar.set_postfix({'loss': f"{total_loss/total:.4f}", 'acc': f"{100.*correct/total:.2f}%"})
    
    return total_loss/len(dataloader), correct/total

def evaluate(model_obj, dataloader, criterion_obj, device_obj, is_frame_level=False):
    model_obj.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Evaluating", leave=False)
        for batch in pbar:
            rgb = batch['rgb'].to(device_obj)
            label = batch['label'].to(device_obj).unsqueeze(1)
            
            if is_frame_level:
                S = rgb.shape[1]
                frame_idx = S // 2 
                outputs = model_obj(rgb[:, frame_idx, :, :, :])
            else:
                fft, audio = batch['fft'].to(device_obj), batch['audio'].to(device_obj)
                outputs = model_obj(rgb, fft, audio)
                
            loss = criterion_obj(outputs, label)
            total_loss += loss.item()
            preds = torch.sigmoid(outputs) > 0.5
            correct += (preds == label).sum().item()
            total += label.size(0)
            pbar.set_postfix({'val_loss': f"{total_loss/total:.4f}", 'val_acc': f"{100.*correct/total:.2f}%"})
    return total_loss/len(dataloader), correct/total

In [ ]:
epochs_s1 = 3
print("STAGE 1: Train ViT (Frame-Level)")
frame_model = FrameLevelViT(model.vit).to(device)
optimizer1 = optim.AdamW(frame_model.parameters(), lr=1e-4, weight_decay=1e-4)

for epoch in range(epochs_s1):
    train_stage(frame_model, train_loader, optimizer1, criterion, device, f"S1 Epoch {epoch+1}", is_frame_level=True)
    evaluate(frame_model, val_loader, criterion, device, is_frame_level=True)

In [ ]:
epochs_s2 = 5
print("STAGE 2: Freeze ViT, Train Temporal/Audio/FFT Context")
for param in model.vit.parameters():
    param.requires_grad = False
    
optimizer2 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)

for epoch in range(epochs_s2):
    train_stage(model, train_loader, optimizer2, criterion, device, f"S2 Epoch {epoch+1}", is_frame_level=False)
    evaluate(model, val_loader, criterion, device, is_frame_level=False)

In [ ]:
epochs_s3 = 5
print("STAGE 3: Fine-Tune Full Multi-Modal Pipeline")
for param in model.vit.parameters():
    param.requires_grad = True
    
optimizer3 = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)

for epoch in range(epochs_s3):
    train_stage(model, train_loader, optimizer3, criterion, device, f"S3 Epoch {epoch+1}", is_frame_level=False)
    evaluate(model, val_loader, criterion, device, is_frame_level=False)

In [ ]:
import os
print("Training Complete! Saving final multi-modal model...")
os.makedirs("../models", exist_ok=True)
torch.save(model.state_dict(), "../models/final_multimodal_model.pth")
print("Model saved to ../models/final_multimodal_model.pth")